# B1 · Qwen-Image MM-DiT (dual stream) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/B1_qwen_image.ipynb)

원본: diffusers `QwenImageTransformer2DModel` (`src/diffusers/models/transformers/transformer_qwenimage.py`,
PyPI `diffusers==0.41.0` = 커밋 `086bf95` 고정. main `af4271c` 는 Neuron 전용 RoPE 분기만 지운 것으로 CPU/CUDA 계산은 같음).
내 구현: `miniarch/models/qwen_image.py`. 가중치 이름은 실제 체크포인트(`Qwen/Qwen-Image` `transformer/`)와 같습니다.
VAE 와 텍스트 인코더(Qwen2.5-VL)는 빼고, transformer 만 비교합니다 (텍스트 입력 = 인코더 폭의 랜덤 hidden).

부품
- dual stream 블록: 이미지·텍스트가 adaLN·q/k/v·출력·MLP 가중치를 따로 갖고, attention 만 [텍스트, 이미지] 를 이어 한 번에 (양방향)
- adaLN: timestep → [shift, scale, gate] × (attn, MLP). 최종층은 [scale, shift] 순서
- QK-norm: head 별 RMSNorm (이미지 `norm_q/k`, 텍스트 `norm_added_q/k`)
- MSRoPE: head_dim 을 [frame | 높이 | 너비] 로 나눈 3축 RoPE. 이미지 위치는 가운데 0 기준, 텍스트는 이미지 대각선 뒤
- 텍스트 padding mask (key 쪽만 가림), 편집용 `zero_cond_t` (참조 이미지 토큰은 t=0 modulation)

축소 config: 2층, d=64 (head 4 × 16), RoPE 축 (4, 6, 6), 텍스트 폭 48, latent 16채널(=4ch × 2×2 patch), 텍스트 7토큰.
- 두 번째 샘플의 텍스트 mask 는 `[1,1,0,1,1,0,0]` (중간 구멍 + 끝 padding)
- edit(CASE): 이미지 2장 (1,3,4)+(1,4,2) → 홀수 높이의 가운데 정렬, frame 위치 0/1, 토큰별 modulation 선택 경로
- t2i / layered 는 자체 점검에서 같은 방식으로 비교 (layered: 4장, 마지막 조건 이미지 frame -1, 추가 t 조건 Embedding)
- 층별로 attention 출력, 두 MLP, 블록 출력(텍스트 쪽), 최종 norm 과 최종 출력을 비교합니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. 학습 동작 확인 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
if not os.environ.get('MINIARCH_SKIP_PIP'):   # 전체 실행 노트북이 한 번에 설치
    !pip -q install -U 'diffusers==0.41.0'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("B1")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('Qwen/Qwen-Image', '^transformer_blocks\\.0\\.', subfolder='transformer')
peek('Qwen/Qwen-Image', '^(img_in|txt_in|txt_norm|time_text_embed|norm_out|proj_out)\\.', subfolder='transformer')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **학습 궤적 == 원본**: 같은 가중치·데이터·optimizer(SGD momentum 0.9, 30 step)로 원본과 내 구현을 나란히 학습해
  step 별 손실과 최종 가중치를 비교합니다 (상대차 ≤ 1e-3). Adam 은 아주 작은 잡음 차이를 키워서 궤적 비교에 쓰지 않습니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다 (LM은 다음 토큰 CE, 그 외는 고정 목표 MSE).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print("원본에서 기울기 없는 파라미터:", g.skipped_ref[:8])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    t = TR.train_parity(ref, pair, inputs, dev)
    if t.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", t.note); report("학습 궤적 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"{t.steps} step  손실 최대 상대차 {t.max_loss_rel:.2e}  최종 가중치 최대 상대차 {t.max_param_rel:.2e} ({t.worst_param})", t.note)
        print("  원본:", " ".join(f"{v:.4g}" for v in t.loss_ref[::6]))
        print("  내것:", " ".join(f"{v:.4g}" for v in t.loss_mine[::6]))
        report("학습 궤적 == 원본 (SGD 30 step)", t.ok, max(t.max_loss_rel, t.max_param_rel))
except Exception:
    traceback.print_exc(limit=4); report("학습 궤적 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

### 원본과 비교할 수 없는 부분

- **수치 미분 (float64)**: 무작위 방향으로 가중치를 ±ε 움직여 잰 손실 변화가 autograd 기울기와 같은지 봅니다.
  원본이 추론 전용이라 기울기를 비교할 수 없는 모델도 이걸로 backward 를 확인합니다.
- **라우터 bias 업데이트 (aux-loss-free)**: 원본 추론 코드에는 없는 학습 전용 규칙입니다. 많이 뽑힌 expert 의 bias 는 내려가고
  적게 뽑힌 쪽은 올라가는지, bias 가 optimizer 에서 빠져 있는지, bias 가 실제로 선택을 바꾸는지 봅니다. 라우터 bias 가 없는 모델은 건너뜁니다.

In [ ]:
try:
    f = TR.directional_grad_check(pair["mine"], pair, inputs, dev)
    print(f"방향 {len(f.rel_errors)}개 상대 오차 {[f'{e:.1e}' for e in f.rel_errors]}  (ε={f.eps})")
    report("수치 미분 == autograd (float64)", f.ok, max(f.rel_errors))
except Exception:
    traceback.print_exc(limit=4); report("수치 미분 == autograd (float64)", False)
try:
    b = TR.router_bias_check(pair["mine"], pair, inputs, dev)
    print("\n".join(b.details))
    report("라우터 bias 업데이트" if b.n_routers else "라우터 bias 업데이트 (라우터 bias 없음 → 건너뜀)", b.ok)
except Exception:
    traceback.print_exc(limit=4); report("라우터 bias 업데이트", False)

## 단계 전용 자체 점검

### 다른 변형도 원본과 같은지 (t2i · layered)

- t2i: Qwen-Image / 2512 처럼 이미지 1장, `zero_cond_t=False`
- layered: Qwen-Image-Layered 처럼 `use_layer3d_rope=True` + `use_additional_t_cond=True`, 이미지 4장 (마지막 = 조건 이미지)

In [ ]:
import dataclasses
for v in ["t2i", "layered"]:
    row = V.cases.run_case(dataclasses.replace(S.CASE, ref=lambda v=v: S.ref(v)))
    print(f"[{v}]", row["status"]); print(row["detail"])
    report(f"원본 동치 ({v})", row["status"].startswith("✅"))

### joint attention map

`attn_probs` [B, H, T+N, T+N] 는 행·열 모두 [텍스트 T | 이미지 N] 순서입니다.
- 행 합 = 1, padding 텍스트 key 열의 확률 = 0
- 이미지 질의가 이미지/텍스트에, 텍스트 질의가 이미지에 주는 평균 확률 질량을 봅니다.

In [ ]:
from miniarch.models import qwen_image as Q
ref, inp = S.ref()
m = Q.verify_pair(ref)["mine"]; V.copy_weights(ref, m); m = m.to(dev).eval()
x = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in inp.items()}
T_ = x["encoder_hidden_states"].shape[1]
mask = x["encoder_hidden_states_mask"].bool()
names = [n for n, _ in m.named_modules() if n.endswith("attn_probs")]
with V.capture(m, names) as st, torch.no_grad():
    m(**x)
for n in names:
    p = st[n]
    err = (p.sum(-1) - 1).abs().max().item()
    pad = max(p[b, :, :, :T_][..., ~mask[b]].abs().max().item() if (~mask[b]).any() else 0.0 for b in range(p.shape[0]))
    ii, it = p[:, :, T_:, T_:].sum(-1).mean().item(), p[:, :, T_:, :T_].sum(-1).mean().item()
    ti = p[:, :, :T_, T_:].sum(-1).mean().item()
    print(f"{n:<36} {tuple(p.shape)}  이미지→이미지 {ii:.3f}  이미지→텍스트 {it:.3f}  텍스트→이미지 {ti:.3f}")
    report(f"{n} 행 합 = 1", err < 1e-5, err)
    report(f"{n} padding key 확률 = 0", pad == 0.0, pad)

### 텍스트 padding · CFG 빈 텍스트 경로

- padding 위치의 텍스트 내용을 바꾸거나, 끝에 padding 토큰을 더 붙여도 이미지 출력이 그대로여야 합니다
  (텍스트 RoPE 위치는 앞에서부터라 유효 토큰 위치가 변하지 않음).
- CFG 의 무조건 경로처럼 텍스트를 전부 가리면 텍스트 내용과 무관해야 하고, 원본도 같은 값을 내야 합니다.

In [ ]:
from miniarch.models import qwen_image as Q
ref, inp = S.ref()
m = Q.verify_pair(ref)["mine"]; V.copy_weights(ref, m); m = m.to(dev).eval(); ref = ref.to(dev).eval()
x = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in inp.items()}
e, mk = x["encoder_hidden_states"], x["encoder_hidden_states_mask"]
g = torch.Generator(device=dev).manual_seed(5)
with torch.no_grad():
    y0 = m(**x)
    e2 = e.clone(); pad = ~mk.bool()
    e2[pad] = torch.randn(e2[pad].shape, generator=g, device=dev) * 3
    d1 = (m(**dict(x, encoder_hidden_states=e2)) - y0).abs().max().item()
    e3 = torch.cat([e, torch.randn(e.shape[0], 3, e.shape[2], generator=g, device=dev)], 1)
    mk3 = torch.cat([mk, torch.zeros(mk.shape[0], 3, dtype=mk.dtype, device=dev)], 1)
    d2 = (m(**dict(x, encoder_hidden_states=e3, encoder_hidden_states_mask=mk3)) - y0).abs().max().item()
    xn = dict(x, encoder_hidden_states_mask=torch.zeros_like(mk))
    yn = m(**xn)
    d3 = (m(**dict(xn, encoder_hidden_states=torch.randn(e.shape, generator=g, device=dev))) - yn).abs().max().item()
    d4 = (ref(**xn).sample - yn).abs().max().item()
    d5 = (yn - y0).abs().max().item()
report("padding 내용 바꿔도 출력 같음", d1 < 1e-5, d1)
report("끝 padding 추가해도 출력 같음", d2 < 1e-4, d2)
report("텍스트 전부 mask → 텍스트 내용 무관", d3 < 1e-5 and bool(torch.isfinite(yn).all()), d3)
report("텍스트 전부 mask: 원본 == 내 구현", d4 < 1e-4, d4)
report("조건/무조건 출력이 다름", d5 > 1e-3, d5)

### timestep · adaLN modulation probe

- `img_mod_probe` / `txt_mod_probe`: [shift1, scale1, gate1, shift2, scale2, gate2] (1 = attention, 2 = MLP).
- 편집 경로(`zero_cond_t`): 이미지 modulation 은 [t, 0] 두 벌이고, 뒤 절반이 timestep=0 으로 돌린 결과의 앞 절반과 같아야 합니다.
- timestep 을 바꾸면 출력이 바뀌어야 합니다.

In [ ]:
from miniarch.models import qwen_image as Q
ref, inp = S.ref()
m = Q.verify_pair(ref)["mine"]; V.copy_weights(ref, m); m = m.to(dev).eval()
x = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in inp.items()}
B_ = x["timestep"].shape[0]
names = [n for n, _ in m.named_modules() if n.endswith(("img_mod_probe", "txt_mod_probe"))]
x0 = dict(x, timestep=torch.zeros_like(x["timestep"]))
with V.capture(m, names) as st, torch.no_grad():
    y = m(**x)
with V.capture(m, names) as st0, torch.no_grad():
    y0 = m(**x0)
for n in names:
    a = st[n]
    sh1, sc1, g1, sh2, sc2, g2 = a[:B_].chunk(6, dim=-1)
    print(f"{n:<34} {tuple(a.shape)}  |scale1| {sc1.abs().mean():.3f}  |gate1| {g1.abs().mean():.3f}  |gate2| {g2.abs().mean():.3f}")
    if n.endswith("img_mod_probe") and m.config.zero_cond_t:
        err = (a[B_:] - st0[n][:B_]).abs().max().item()
        report(f"{n} 참조 이미지 = t=0 modulation", a.shape[0] == 2 * B_ and err < 1e-5, err)
d = (y - y0).abs().max().item()
report("timestep 바꾸면 출력이 바뀜", d > 1e-3, d)

### ablation 스위치

부품별 비교 실험에 쓸 스위치를 하나씩 끄고, forward 가 유한값이고 출력이 실제로 바뀌는지 확인합니다.

In [ ]:
import dataclasses
from miniarch.models import qwen_image as Q
ref, inp = S.ref()
m = Q.verify_pair(ref)["mine"]; V.copy_weights(ref, m); m = m.to(dev).eval()
x = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in inp.items()}
with torch.no_grad():
    y = m(**x)
for sw in ["use_qk_norm", "use_rope", "center_rope", "use_text_mask", "joint_attention", "use_txt_norm"]:
    m2 = Q.QwenImage(dataclasses.replace(m.config, **{sw: False})).to(dev).eval()
    m2.load_state_dict(m.state_dict())
    with torch.no_grad():
        out = m2(**x)
    d = (out - y).abs().max().item()
    report(f"{sw}=False forward (출력 변화)", bool(torch.isfinite(out).all()) and d > 1e-5, d)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/B1_qwen_image.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_B1_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")